# NASA Ötegezegen Veri Analizi

**Hazırlayan:** Fatmanur Güven

## Projenin amacı

Bu projede NASA ötegezegen veri seti incelenecektir. Gezegenlerin
keşif yılları, keşif yöntemleri ve fiziksel özellikleri analiz edilecektir.

Araştırma sorularından biri, yarıçapı Dünya yarıçapının 0,8 ile 1,25
katı arasında olan kaç gezegen bulunduğudur.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("NASA_Otegezegen_Veri_Seti.csv")

print("Satır ve sütun sayısı:", df.shape)
display(df.head())

In [ ]:
display(df.head())

In [ ]:
print("Satır ve sütun sayısı:", df.shape)
print("Satır sayısı:", df.shape[0])
print("Sütun sayısı:", df.shape[1])

In [ ]:
df.info()

In [ ]:
yaricap = df["pl_rade"].dropna().to_numpy()

print("Ölçümü bulunan gezegen sayısı:", yaricap.size)
print("Ortalama yarıçap:", round(np.mean(yaricap), 2))
print("Medyan yarıçap:", round(np.median(yaricap), 2))
print("En küçük yarıçap:", round(np.min(yaricap), 2))
print("En büyük yarıçap:", round(np.max(yaricap), 2))

In [ ]:
print("Sütunlara göre eksik değer sayısı:")
print(df.isnull().sum())
print("\nTekrarlanan satır sayısı:", df.duplicated().sum())

In [ ]:
display(df.describe())

In [ ]:
print("Keşif yöntemlerine göre gezegen sayıları:")
display(df["discoverymethod"].value_counts())

## 5. Veri Temizleme

Bu bölümde eksik veriler ve tekrarlanan satırlar incelenecektir.
Eksik fiziksel ölçümler tahmin edilerek doldurulmayacak; ilgili
hesaplamalarda eksik olan değerler dışarıda bırakılacaktır.

In [ ]:
# Her sütundaki eksik değer sayısı
print("Eksik değer sayıları:")
print(df.isnull().sum())

# Birbirinin tamamen aynı olan satırları kontrol et
print("\nTekrarlanan satır sayısı:", df.duplicated().sum())

In [ ]:
# Orijinal veriyi koruyarak analiz için bir kopya oluştur
df_temiz = df.drop_duplicates().copy()

print("Temizleme öncesi satır sayısı:", len(df))
print("Temizleme sonrası satır sayısı:", len(df_temiz))

## 6. NumPy ile Gezegen Yarıçaplarının İncelenmesi

Gezegen yarıçapları Dünya yarıçapı cinsinden verilmiştir.
Yarıçapı bilinmeyen gezegenler bu hesaplamaya dahil edilmemiştir.

In [ ]:
yaricap = df_temiz["pl_rade"].dropna().to_numpy()

print("Yarıçapı bilinen gezegen sayısı:", len(yaricap))
print("Ortalama yarıçap:", round(np.mean(yaricap), 2))
print("Medyan yarıçap:", round(np.median(yaricap), 2))
print("En küçük yarıçap:", round(np.min(yaricap), 2))
print("En büyük yarıçap:", round(np.max(yaricap), 2))

## 7. Dünya Boyutundaki Gezegenler

Bu çalışmada yarıçapı Dünya yarıçapının 0,8 ile 1,25 katı arasında
olan gezegenler "Dünya boyutunda" olarak sınıflandırılmıştır.

Bu sınırlar analiz için seçilmiştir. Dünya boyutunda olmak,
gezegenin yaşanabilir olduğunu göstermez.

In [ ]:
dunya_boyutunda = df_temiz[
    (df_temiz["pl_rade"] >= 0.8) &
    (df_temiz["pl_rade"] <= 1.25)
].copy()

print("Dünya boyutundaki gezegen sayısı:", len(dunya_boyutunda))

display(
    dunya_boyutunda[
        ["pl_name", "disc_year", "discoverymethod", "pl_rade"]
    ].head(10)
)

## 8. Yakındaki Gezegenler

Bu bölümde Dünya'ya uzaklığı 100 parsek veya daha az olan
gezegenler incelenmektedir.

In [ ]:
yakin_gezegenler = df_temiz[
    df_temiz["sy_dist"] <= 100
].copy()

print("100 parsek veya daha yakındaki gezegen sayısı:", len(yakin_gezegenler))

display(
    yakin_gezegenler[
        ["pl_name", "hostname", "sy_dist", "discoverymethod"]
    ].sort_values("sy_dist").head(10)
)

## 9. Gezegenleri Boyutlarına Göre Sınıflandırma

Bu projede yarıçapı bilinen gezegenler analiz amacıyla üç gruba
ayrılmıştır: Dünya'dan küçük, Dünya boyutunda ve Dünya'dan büyük.
Yarıçapı bilinmeyenler ayrıca gösterilmiştir.

In [ ]:
kosullar = [
    df_temiz["pl_rade"] < 0.8,
    df_temiz["pl_rade"].between(0.8, 1.25),
    df_temiz["pl_rade"] > 1.25
]

etiketler = [
    "Dünya'dan küçük",
    "Dünya boyutunda",
    "Dünya'dan büyük"
]

df_temiz["boyut_sinifi"] = np.select(
    kosullar,
    etiketler,
    default="Yarıçap bilinmiyor"
)

print("Boyut sınıflarına göre gezegen sayıları:")
display(df_temiz["boyut_sinifi"].value_counts())

## 10. Keşif Yöntemlerine Göre Özet Tablosu

Gezegenler keşif yöntemlerine göre gruplandırılarak her yöntemdeki
gezegen sayısı ve yarıçapı bilinen gezegenlerin ortalama yarıçapı
hesaplanmıştır.

In [ ]:
yontem_ozeti = (
    df_temiz
    .groupby("discoverymethod")
    .agg(
        gezegen_sayisi=("pl_name", "count"),
        yaricapi_bilinen=("pl_rade", "count"),
        ortalama_yaricap=("pl_rade", "mean")
    )
    .sort_values("gezegen_sayisi", ascending=False)
)

display(yontem_ozeti.round(2))

## 11. Veri Görselleştirme

### Grafik 1: Yıllara Göre Keşfedilen Gezegen Sayısı

In [ ]:
yillik_kesif = df_temiz["disc_year"].value_counts().sort_index()

plt.figure(figsize=(12, 5))
plt.plot(yillik_kesif.index, yillik_kesif.values, marker="o")

plt.title("Yıllara Göre Keşfedilen Ötegezegen Sayısı")
plt.xlabel("Keşif yılı")
plt.ylabel("Gezegen sayısı")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Grafik yorumu:**

Veri setinde en fazla gezegen keşfinin yapıldığı yıl 2016'dır.
Bu yılda 1.504 gezegen kaydedilmiştir. Keşif sayıları yıllara göre
değişmektedir; bu değişimin nedenleri için gözlem görevleri ayrıca
incelenmelidir.

In [ ]:
print("En fazla keşif yapılan yıl:", yillik_kesif.idxmax())
print("O yıl keşfedilen gezegen sayısı:", yillik_kesif.max())

### Grafik 2: Keşif Yöntemlerine Göre Gezegen Sayısı

In [ ]:
yontem_sayilari = df_temiz["discoverymethod"].value_counts()

plt.figure(figsize=(12, 6))
sns.barplot(
    x=yontem_sayilari.values,
    y=yontem_sayilari.index,
    color="steelblue"
)

plt.title("Keşif Yöntemlerine Göre Ötegezegen Sayısı")
plt.xlabel("Gezegen sayısı")
plt.ylabel("Keşif yöntemi")
plt.tight_layout()
plt.show()

**Grafik yorumu:**

Veri setinde en sık kullanılan keşif yöntemi Transit yöntemidir.
Transit ile 4.800 gezegen, Radial Velocity yöntemiyle ise 1.204
gezegen kaydedilmiştir. Bu dağılım, veri setindeki keşif yöntemlerini
gösterir; evrendeki bütün gezegenlerin dağılımını göstermez.

### Grafik 3: Gezegen Yarıçaplarının Dağılımı

Grafiğin daha okunaklı olması için yarıçapı 20 Dünya yarıçapından
büyük olan gezegenler bu grafikte gösterilmemiştir. Bu değerler
veri setinden silinmemiştir.

In [ ]:
grafik_yaricap = df_temiz.loc[
    df_temiz["pl_rade"].between(0, 20),
    "pl_rade"
]

plt.figure(figsize=(10, 5))
plt.hist(grafik_yaricap, bins=30, color="mediumpurple", edgecolor="white")

plt.title("Ötegezegen Yarıçaplarının Dağılımı")
plt.xlabel("Yarıçap (Dünya yarıçapı)")
plt.ylabel("Gezegen sayısı")
plt.tight_layout()
plt.show()

print("Grafikte gösterilen gezegen sayısı:", len(grafik_yaricap))
print("Yarıçapı bilinen toplam gezegen sayısı:", df_temiz["pl_rade"].count())

**Grafik yorumu:**

Grafikte gezegen yarıçapları özellikle yaklaşık 2,27–2,93 Dünya
yarıçapı aralığında yoğunlaşmaktadır. Grafikte yalnızca yarıçapı
20 Dünya yarıçapına kadar olan gezegenler gösterilmiştir; daha büyük
gezegenler bu grafikte yer almamaktadır.

### Grafik 4: Gezegenlerin Boyut Sınıfları

In [ ]:
boyut_sayilari = df_temiz["boyut_sinifi"].value_counts()

plt.figure(figsize=(10, 5))
sns.barplot(
    x=boyut_sayilari.index,
    y=boyut_sayilari.values,
    color="teal"
)

plt.title("Gezegenlerin Boyut Sınıfları")
plt.xlabel("Boyut sınıfı")
plt.ylabel("Gezegen sayısı")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

display(boyut_sayilari)

**Grafik yorumu:**

Veri setinde 517 gezegen, bu proje için belirlenen Dünya boyutu
aralığındadır. Yarıçapı bilinmeyen 50 gezegen ayrı bir sınıfta
gösterilmiştir; bu gezegenlerin boyutu hakkında çıkarım yapılamaz.

## 12. Betimsel İstatistik ve Korelasyon

Betimsel istatistikler sayısal değişkenlerin dağılımını özetler.
Korelasyon ise iki sayısal değişken arasındaki doğrusal ilişkinin
yönü ve gücü hakkında bilgi verir. Korelasyon, tek başına
neden-sonuç ilişkisi göstermez.

In [ ]:
sayisal_sutunlar = [
    "pl_rade",
    "pl_bmasse",
    "pl_orbper",
    "pl_eqt",
    "sy_dist"
]

display(df_temiz[sayisal_sutunlar].describe().round(2))

In [ ]:
korelasyon = df_temiz[sayisal_sutunlar].corr()

display(korelasyon.round(2))

In [ ]:
plt.figure(figsize=(9, 6))

sns.heatmap(
    korelasyon,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.title("Sayısal Değişkenler Arasındaki Korelasyon")
plt.tight_layout()
plt.show()

**Grafik yorumu:**

Gezegen yarıçapı (pl_rade) ile gezegen kütlesi (pl_bmasse)
arasındaki korelasyon yaklaşık 0,45'tir. Bu değer, veri setinde
iki değişken arasında orta düzeyde pozitif bir doğrusal ilişki
olduğunu gösterir; tek başına neden-sonuç ilişkisi kurdurmaz.

In [ ]:
print("Toplam gezegen sayısı:", len(df_temiz))
print("Dünya boyutundaki gezegen sayısı:", len(dunya_boyutunda))
print("Yarıçapı bilinen gezegen sayısı:", df_temiz["pl_rade"].count())
print("Yarıçapı bilinmeyen gezegen sayısı:", df_temiz["pl_rade"].isnull().sum())
print("En yaygın keşif yöntemi:", yontem_sayilari.idxmax())
print("En fazla keşif yapılan yıl:", yillik_kesif.idxmax())

## 13. Sonuç ve Öneriler

1. Veri setinde 6.473 gezegen kaydı bulunmaktadır. Bunların 517
   tanesi, bu projede kullanılan 0,8–1,25 Dünya yarıçapı aralığındadır.
   Yarıçapı bilinmeyen 50 kayıt bu sınıflandırmada ayrı tutulmuştur.

2. Transit, veri setindeki en yaygın keşif yöntemidir. Farklı keşif
   yöntemlerinin hangi gezegenleri daha kolay tespit edebildiği
   araştırılarak bu bulgu daha ayrıntılı incelenebilir.

3. Keşiflerin yıllara göre dağılımı farklılık göstermektedir.
   En fazla keşif yapılan yıl 2016 olarak bulunmuştur. Bu değişimin
   nedenlerini incelemek için o yıllardaki gözlem görevleri ayrıca
   araştırılabilir.

Bu analiz, kullanılan veri setindeki kayıtlarla sınırlıdır. Gezegenin
Dünya boyutunda olması tek başına yaşanabilir olduğunu göstermez.